# 🥁 Drum Copy — Google Colab 実行ノートブック

楽曲ファイル（wav / mp3）からドラムパートを分離し、MIDIファイルとして出力します。

```
入力音声 → [Demucs: stem分離] → ドラムwav → [omnizart: 自動採譜] → MIDI
```

## 実行前の準備

1. **GPUランタイムを有効化**  
   「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU（T4）」

2. **入力音声を Google Drive に配置**  
   `マイドライブ/drum-copy_data/input/` に処理したい wav / mp3 ファイルを入れる

3. **上から順にセルを実行**（「ランタイム」→「すべてのセルを実行」でも可）

> 出力 MIDI は `マイドライブ/drum-copy_data/output/` に保存されます。  
> ソースコードは `/content/drum-copy`（Colabローカル）に git clone されます。

## 1. GPU 確認

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("⚠️  GPU が検出されません。")
    print("「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU」を設定してください。")
    print("CPU でも実行できますが、処理に数十分かかる場合があります。")

## 2. パッケージインストール

> ランタイムが変わるたびに再実行が必要です。初回は数分かかります。

In [ ]:
# omnizart は vamp / madmom に依存しているが Colab 環境では別途処理が必要なためパッチインストール
import subprocess, sys, re, shutil
from pathlib import Path

def pip_run(*pkgs, flags=None):
    cmd = [sys.executable, "-m", "pip", "install"] + list(pkgs) + (flags or [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        out = r.stdout + r.stderr
        print(out[:3000])
        if len(out) > 3000:
            print("...[中略]...")
            print(out[-2000:])
        raise subprocess.CalledProcessError(r.returncode, cmd)

def git_clone(url, dest):
    r = subprocess.run(["git", "clone", "--depth=1", url, str(dest)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"git clone 失敗 (exit {r.returncode}): {url}\n"
                           "リポジトリが存在しないか、ネットワークエラーの可能性があります。")

print("1/7  numpy をピン留め（2.x 非互換回避）...")
pip_run("numpy<2.0", flags=["-q"])

print("2/7  demucs / soundfile...")
pip_run("demucs", "soundfile", flags=["-q"])

print("3/7  setuptools / wheel（Python 3.12 の distutils 削除対策）...")
pip_run("setuptools<70", "wheel", flags=["-q"])

print("4/7  Cython（madmom ビルドに必要）...")
pip_run("Cython<3", flags=["-q"])

print("5/7  madmom をソースからビルド（Python 3.12 / Colab 互換パッチ適用）...")
_madmom_src = Path("/content/_madmom_src")
if _madmom_src.exists():
    shutil.rmtree(_madmom_src)
git_clone("https://github.com/CPJKU/madmom.git", _madmom_src)
# Python 3.12 で削除された distutils を setuptools で置換
_setup_py = _madmom_src / "setup.py"
_txt = _setup_py.read_text()
_txt = _txt.replace("from distutils.core import",            "from setuptools import")
_txt = _txt.replace("from distutils.extension import",       "from setuptools.extension import")
_txt = _txt.replace("from distutils.command.build_ext import", "from setuptools.command.build_ext import")
_setup_py.write_text(_txt)
pip_run(str(_madmom_src), flags=["-q", "--no-build-isolation"])

print("6/7  omnizart をクローン & パッチ...")
src_dir = Path("/content/_omnizart_src")
if src_dir.exists():
    print(f"  既存ディレクトリを削除: {src_dir}")
    shutil.rmtree(src_dir)
git_clone("https://github.com/Music-and-Culture-Technology-Lab/omnizart.git", src_dir)

# setup.py から vamp / madmom（別途インストール済み）を除去
setup_py = src_dir / "setup.py"
if setup_py.exists():
    src = setup_py.read_text()
    src = re.sub(r"""['"](vamp|madmom)[^'"]*['"],?\s*""", "", src)
    setup_py.write_text(src)

# pyproject.toml がある場合も対応
pyproject = src_dir / "pyproject.toml"
if pyproject.exists():
    src = pyproject.read_text()
    src = re.sub(r"""['"](vamp|madmom)[^'"]*['"]""", '""', src)
    pyproject.write_text(src)

print("7/7  omnizart をインストール...")
pip_run(str(src_dir), flags=["-q", "--no-build-isolation"])

print("✅ インストール完了")

## 3. コードの取得

GitHubリポジトリを `/content/drum-copy` に clone します（再実行時は `git pull` で最新化）。

In [ ]:
import subprocess, sys
from pathlib import Path

# ── リポジトリ URL（自分のリポジトリに合わせて変更） ─────────────────
REPO_URL  = "https://github.com/Hukumaro/drum-copy.git"
CODE_ROOT = Path("/content/drum-copy")
# ────────────────────────────────────────────────────────────────────

if CODE_ROOT.exists():
    r = subprocess.run(["git", "-C", str(CODE_ROOT), "pull"],
                       capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_ROOT)], check=True)

sys.path.insert(0, str(CODE_ROOT))
print(f"✅ コードパス: {CODE_ROOT}")

## 4. Google Drive マウント

入出力データの保存先として Google Drive をマウントします。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive マウント完了")

## 5. パス設定

データの保存先（Google Drive）を設定します。  
**通常はここを変更する必要はありません。**

In [ ]:
from backends.colab import ColabBackend

# ── 設定 ────────────────────────────────────────────────────────────
MODEL     = "htdemucs"    # Demucs モデル（htdemucs / htdemucs_ft / mdx_extra など）
OVERWRITE = False         # True にすると既存の MIDI を上書き

# データ保存先（Drive）。変更する場合はここを修正。
DATA_ROOT = "/content/drive/MyDrive/drum-copy_data"
# ────────────────────────────────────────────────────────────────────

backend = ColabBackend(
    data_root=DATA_ROOT,
    use_gdrive=False,   # Drive は上のセルでマウント済み
    tmp_dir="/content/tmp/drum-copy",
)
backend.setup()
paths = backend.get_paths()

SUPPORTED = {".wav", ".mp3"}
audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

print(f"コードパス      : {CODE_ROOT}")
print(f"入力ディレクトリ : {paths.input_dir}")
print(f"出力ディレクトリ : {paths.output_dir}")
print(f"一時ディレクトリ : {paths.tmp_dir}")
print(f"モデル          : {MODEL}")
print()
print(f"入力ファイル ({len(audio_files)} 件):")
for f in audio_files:
    print(f"  - {f.name}")
if not audio_files:
    print(f"⚠️  {paths.input_dir} に音声ファイルが見つかりません。")

### (オプション) Google Drive を使わず直接アップロードする場合

下のセルのコメントを外して実行すると、PC からファイルをアップロードできます。

In [ ]:
# from google.colab import files as colab_files
# import shutil
#
# uploaded = colab_files.upload()
# for fname in uploaded:
#     dest = paths.input_dir / fname
#     shutil.move(fname, dest)
#     print(f"保存: {dest}")
#
# # アップロード後に audio_files を更新
# audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

## 6. パイプライン実行

- **Step 1** : Demucs でドラムトラックを分離 → `tmp/` に保存  
- **Step 2** : omnizart でドラムを採譜 → `output/` に MIDI 保存

> 1曲あたりの目安: GPU(T4) で 2〜5分、CPU では 20〜60分

In [ ]:
import logging
import shutil

from pipeline.stem_separation.separator import separate
from pipeline.transcription.transcriber import transcribe

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("drum_copy")

if not audio_files:
    raise FileNotFoundError(f"入力ファイルがありません: {paths.input_dir}")

results = []
for audio_path in audio_files:
    midi_out = paths.output_dir / (audio_path.stem + ".mid")

    if midi_out.exists() and not OVERWRITE:
        logger.info("スキップ（既存）: %s", midi_out.name)
        results.append((audio_path.name, str(midi_out), "SKIP"))
        continue

    try:
        logger.info("=== Step 1: Stem 分離 — %s ===", audio_path.name)
        drums_wav = separate(audio_path, paths.tmp_dir, model=MODEL)

        logger.info("=== Step 2: MIDI 変換 — %s ===", drums_wav.name)
        midi_path = transcribe(drums_wav, paths.output_dir)

        if midi_path.stem != audio_path.stem:
            renamed = paths.output_dir / (audio_path.stem + ".mid")
            midi_path.rename(renamed)
            midi_path = renamed

        stem_tmp = paths.tmp_dir / audio_path.stem
        if stem_tmp.exists():
            shutil.rmtree(stem_tmp)

        results.append((audio_path.name, str(midi_path), "OK"))
        logger.info("完了: %s", midi_path)

    except Exception as exc:
        logger.error("失敗 %s: %s", audio_path.name, exc)
        results.append((audio_path.name, "", f"ERROR: {exc}"))

print("\n─── 結果 ─────────────────────────────────────")
for src, dst, status in results:
    icon = "✅" if status in ("OK", "SKIP") else "❌"
    print(f"{icon}  {src}  →  {dst or status}")

## 7. MIDI ダウンロード（オプション）

出力 MIDI は Google Drive の `output/` に自動保存されています。  
Colab から直接ダウンロードしたい場合は下のセルを実行してください。

In [ ]:
from google.colab import files as colab_files

midi_files = list(paths.output_dir.glob("*.mid"))
if midi_files:
    for f in midi_files:
        print(f"ダウンロード中: {f.name}")
        colab_files.download(str(f))
else:
    print("出力 MIDI ファイルが見つかりません。")